# Qwen3-ASR Thai Engram — Step 300→1000 Scaling Evaluation
### Local ROCm / RX 9070 XT

This notebook continues **after training has already finished**.

It does **not train anything**.

Flow:

```text
ROCm RX 9070 XT
→ Load frozen Qwen3-ASR-0.6B
→ Rebuild the exact Layer-2 Engram architecture
→ Recreate the same fixed 300-sample held-out set
→ Base inference (once)
→ For each checkpoint step in [300, 500, 750, 1000]:
    reload that step's Engram weights → run inference → CER comparison
→ Save per-step results + a combined scaling summary
```

The evaluation configuration is intentionally identical to the scaling notebook:

- Preset A
- Layer 2
- Bigram buckets = 10,000
- Trigram buckets = 2,000
- Engram dim = 512
- 300 held-out samples
- Dataset source offset = 250,000
- Qwen backbone frozen 100%
- Checkpoint steps evaluated: 300, 500, 750, 1000

No Google Drive mount and no optimizer/scheduler/training loop.

## 0. ROCm environment

Use the ROCm PyTorch environment already installed for the RX 9070 XT.

**Do not install a CUDA PyTorch wheel.**

PyTorch ROCm intentionally uses APIs such as `torch.cuda.*` and `device="cuda"`; the backend is still HIP/ROCm.


In [18]:
# Install only non-PyTorch dependencies.
import sys, subprocess, importlib.metadata as _metadata

deps = [
    "qwen-asr",
    "datasets",
    "soundfile",
    "librosa",
    "jiwer",
    "accelerate",
    "huggingface_hub",
    "pandas",
    "tqdm",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U",
    "--upgrade-strategy", "only-if-needed",
    *deps
])

print("torch:", _metadata.version("torch"))
print("qwen-asr:", _metadata.version("qwen-asr"))


  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.29.0-py3-none-any.whl.metadata (16 kB)
torch: 2.9.1+gitunknown
qwen-asr: 0.0.6


## 1. Evaluation configuration

Put the final checkpoint on the local machine and either:

```bash
export ENGRAM_STEP1000_CHECKPOINT=/absolute/path/engram_step_001000.pt
```

or place it under:

```text
Qwen3ASR_Thai_Engram_ROCm/checkpoints/engram_step_001000.pt
```


In [19]:
import os, json, math, random, re, io, inspect, types, subprocess
from pathlib import Path
from contextlib import contextmanager
import numpy as np
import torch
import torch.nn as nn
import soundfile as sf
import librosa
import pandas as pd
from datasets import load_dataset, Audio
from jiwer import cer
from tqdm.auto import tqdm
from qwen_asr import Qwen3ASRModel

MODEL_ID = "Qwen/Qwen3-ASR-0.6B"
DATASET_NAME = "CMKL/Porjai-Thai-voice-dataset-central"
SEED = 42

ENGRAM_LAYERS = [2]
ENGRAM_BUCKETS = {2: 10_000, 3: 2_000}
ENGRAM_DIM = 512
ENGRAM_HEADS = 16
ENGRAM_KERNEL = 4

MIN_AUDIO_SEC = 1.0
MAX_AUDIO_SEC = 8.0
MAX_NEW_TOKENS = 192
GENERATION_USE_CACHE = False

SCALING_EVAL_SOURCE_SKIP = 250_000
SCALING_EVAL_SAMPLES = 300
CHECKPOINT_STEPS = [300, 500, 750, 1000]

PROJECT_ROOT = Path(
    os.environ.get(
        "ENGRAM_LOCAL_ROOT",
        str(Path.cwd() / "Qwen3ASR_Thai_Engram_ROCm")
    )
).expanduser().resolve()
CKPT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_CKPT_DIR = Path(
    os.environ.get("ENGRAM_LOCAL_CKPT_DIR", "/home/cepheusn/Developments/Engram-qwenASR/checkpoints")
).expanduser().resolve()

CHECKPOINT_PATHS = {
    step: LOCAL_CKPT_DIR / f"engram_step_{step:06d}.pt"
    for step in CHECKPOINT_STEPS
}
missing_ckpts = [str(p) for p in CHECKPOINT_PATHS.values() if not p.is_file()]
if missing_ckpts:
    raise FileNotFoundError(
        "Missing checkpoint file(s):\n" + "\n".join(missing_ckpts) +
        "\nSet ENGRAM_LOCAL_CKPT_DIR or copy the engram_step_*.pt files into CKPT_DIR."
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Checkpoints:")
for step in CHECKPOINT_STEPS:
    print(f"  step {step}: {CHECKPOINT_PATHS[step]}")
print("Output dir:", OUTPUT_DIR)
print("Evaluation source skip:", SCALING_EVAL_SOURCE_SKIP)
print("Evaluation samples:", SCALING_EVAL_SAMPLES)


Checkpoint: /home/cepheusn/Developments/Engram-qwenASR/checkpoints/engram_step_001000.pt
Output dir: /home/cepheusn/Developments/Engram-qwenASR/Qwen3ASR_Thai_Engram_ROCm/outputs
Evaluation source skip: 250000
Evaluation samples: 300


## 2. Load frozen Qwen3-ASR on ROCm


In [20]:
import gc, inspect, io, math, os, random, re, json, types, subprocess
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Any, Dict, List

import numpy as np
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Audio
from qwen_asr import Qwen3ASRModel
from transformers import GenerationConfig

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ROCm validation.
if torch.version.hip is None:
    raise RuntimeError(
        "This local notebook requires ROCm/HIP PyTorch. "
        f"Detected torch={torch.__version__}, torch.version.hip={torch.version.hip!r}. "
        "Install/activate a ROCm PyTorch environment; do not use a CUDA wheel."
    )
if not torch.cuda.is_available():
    raise RuntimeError("ROCm PyTorch is installed but no AMD GPU is visible.")

device = torch.device("cuda")  # PyTorch ROCm intentionally uses the torch.cuda API.
gpu_name = torch.cuda.get_device_name(0)
hip_version = torch.version.hip
try:
    gpu_props = torch.cuda.get_device_properties(0)
    total_vram_gb = gpu_props.total_memory / 2**30
except Exception:
    gpu_props = None
    total_vram_gb = float("nan")

# RX 9070 XT supports BF16; prefer the runtime capability check where available.
try:
    USE_BF16 = bool(torch.cuda.is_bf16_supported())
except Exception:
    USE_BF16 = True

# Override with ENGRAM_FORCE_FP16=1 if a specific ROCm/qwen-asr build has a BF16 issue.
if os.environ.get("ENGRAM_FORCE_FP16", "0").strip() == "1":
    USE_BF16 = False

MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("PyTorch       :", torch.__version__)
print("ROCm / HIP    :", hip_version)
print("GPU           :", gpu_name)
print("VRAM (GiB)    :", round(total_vram_gb, 2))
print("Qwen dtype    :", MODEL_DTYPE)
print("torch.cuda API: expected on ROCm")
try:
    rocminfo = subprocess.run(
        ["rocminfo"], capture_output=True, text=True, timeout=10, check=False
    ).stdout
    gfx = sorted(set(re.findall(r"gfx\d+", rocminfo)))
    print("rocminfo arch :", gfx[:8] if gfx else "not detected")
    if gfx and "gfx1201" not in gfx:
        print("WARNING: RX 9070 XT is expected to expose gfx1201; detected:", gfx)
except Exception as exc:
    print("rocminfo check skipped:", repr(exc))

print("from_pretrained:", inspect.signature(Qwen3ASRModel.from_pretrained))

try:
    asr = Qwen3ASRModel.from_pretrained(
        MODEL_ID, dtype=MODEL_DTYPE, device_map=None
    )
except TypeError:
    asr = Qwen3ASRModel.from_pretrained(
        MODEL_ID, torch_dtype=MODEL_DTYPE, device_map=None
    )

model = asr.model.to(device)
processor, tokenizer = asr.processor, asr.processor.tokenizer

def patch_outer_forward(instance):
    if getattr(instance, "_thai_engram_instance_forward_patched", False):
        return
    if not hasattr(instance, "thinker") or not hasattr(instance.thinker, "forward"):
        raise RuntimeError("Incompatible qwen-asr: thinker.forward is unavailable")
    def _forward(self, input_ids=None, attention_mask=None, input_features=None,
                 feature_attention_mask=None, labels=None, **kwargs):
        # Call the module, not its .forward method, so registered thinker pre-hooks run.
        return self.thinker(input_ids=input_ids, attention_mask=attention_mask,
            input_features=input_features, feature_attention_mask=feature_attention_mask,
            labels=labels, **kwargs)
    instance.forward = types.MethodType(_forward, instance)
    instance._thai_engram_instance_forward_patched = True

patch_outer_forward(model)
print("thinker.forward:", inspect.signature(model.thinker.forward))
print("model.generate:", inspect.signature(model.generate))
try:
    model.generation_config = GenerationConfig.from_model_config(model.config)
except Exception as exc:
    print("generation_config warning:", exc)
for p in model.parameters():
    p.requires_grad_(False)
print(f"Frozen Qwen parameters: {sum(p.numel() for p in model.parameters()):,}")


PyTorch       : 2.9.1
ROCm / HIP    : 7.1.52802-9999
GPU           : AMD Radeon RX 9070 XT
VRAM (GiB)    : 15.92
Qwen dtype    : torch.bfloat16
torch.cuda API: expected on ROCm
rocminfo arch : ['gfx12', 'gfx1201']
from_pretrained: (pretrained_model_name_or_path: str, forced_aligner: str | None = None, forced_aligner_kwargs: Dict[str, Any] | None = None, max_inference_batch_size: int = 32, max_new_tokens: int | None = 512, **kwargs) -> 'Qwen3ASRModel'
thinker.forward: (input_ids=None, input_features=None, attention_mask=None, feature_attention_mask=None, audio_feature_lengths=None, position_ids=None, past_key_values=None, inputs_embeds=None, rope_deltas=None, labels=None, use_cache=None, cache_position=None, **kwargs) -> tuple | qwen_asr.core.transformers_backend.modeling_qwen3_asr.Qwen3ASRThinkerCausalLMOutputWithPast
model.generate: (input_ids: torch.Tensor | None = None, max_new_tokens: int = 4096, eos_token_id: int | list[int] = [151645, 151643], **kwargs)
Frozen Qwen parameters: 78

## 3. Rebuild the exact Engram architecture


In [21]:
if not hasattr(model, "thinker") or not hasattr(model.thinker, "model"):
    raise RuntimeError("Qwen thinker/model path is unavailable")
decoder_layers = model.thinker.model.layers
decoder_name = "thinker.model.layers"
if len(decoder_layers) < 20:
    raise RuntimeError(f"Unexpected text decoder depth: {len(decoder_layers)}")

def infer_hidden_size_for(target_model, layers):
    for obj in (layers[0], getattr(layers[0], "self_attn", None), getattr(layers[0], "mlp", None)):
        for attr in ("hidden_size", "embed_dim"):
            if obj is not None and getattr(obj, attr, None) is not None:
                return int(getattr(obj, attr))
    cfg = getattr(target_model.thinker.config, "text_config", None)
    if cfg is not None and getattr(cfg, "hidden_size", None) is not None:
        return int(cfg.hidden_size)
    raise RuntimeError("Could not infer Qwen hidden size")

HIDDEN_SIZE = infer_hidden_size_for(model, decoder_layers)
print("Exact decoder path:", decoder_name)
print("Decoder layers:", len(decoder_layers), "hidden size:", HIDDEN_SIZE)
print("Layer class:", decoder_layers[0].__class__.__name__)


Exact decoder path: thinker.model.layers
Decoder layers: 28 hidden size: 1024
Layer class: Qwen3ASRThinkerTextDecoderLayer


In [22]:
class ThaiNgramEngram(nn.Module):
    """Token-ID N-gram memory; it never normalizes or strips Unicode."""
    def __init__(self, hidden_size, buckets, memory_dim=512, num_heads=16,
                 kernel_size=4, pad_id=0, seed=42):
        super().__init__()
        if memory_dim % num_heads:
            raise ValueError("memory_dim must be divisible by num_heads")
        self.hidden_size = int(hidden_size)
        self.buckets = {int(k): int(v) for k, v in buckets.items()}
        self.orders = sorted(self.buckets)
        self.memory_dim, self.num_heads = int(memory_dim), int(num_heads)
        self.head_dim, self.pad_id = self.memory_dim // self.num_heads, int(pad_id)
        self.tables = nn.ModuleDict()
        for n in self.orders:
            for h in range(self.num_heads):
                self.tables[f"n{n}_h{h}"] = nn.Embedding(self.buckets[n], self.head_dim)
        gen = torch.Generator(device="cpu"); gen.manual_seed(seed)
        self.register_buffer("hash_seeds", torch.randint(
            1009, 2_000_000_000, (max(self.orders)+1, self.num_heads, max(self.orders)),
            generator=gen, dtype=torch.int64) | 1, persistent=True)
        merged = len(self.orders) * self.memory_dim
        self.key_proj, self.value_proj = nn.Linear(merged, self.hidden_size), nn.Linear(merged, self.hidden_size)
        self.key_norm = nn.RMSNorm(self.hidden_size)
        self.query_norm = nn.RMSNorm(self.hidden_size)
        dilation = max(self.orders)
        self.short_conv = nn.Conv1d(self.hidden_size, self.hidden_size, kernel_size=kernel_size,
                                    groups=self.hidden_size, dilation=dilation,
                                    padding=(kernel_size-1)*dilation, bias=False)
        nn.init.zeros_(self.value_proj.weight); nn.init.zeros_(self.value_proj.bias)
        nn.init.zeros_(self.short_conv.weight)

    def _shift_right(self, ids, k):
        if k == 0: return ids
        pad = torch.full((ids.shape[0], k), self.pad_id, dtype=ids.dtype, device=ids.device)
        return torch.cat([pad, ids[:, :-k]], dim=1)

    def _memory_for_order(self, ids, n):
        shifted = [self._shift_right(ids, k) for k in range(n)]
        vectors = []
        for h in range(self.num_heads):
            mixed = torch.zeros_like(ids, dtype=torch.int64)
            for k in range(n):
                mixed = torch.bitwise_xor(mixed, shifted[k].to(torch.int64) * self.hash_seeds[n, h, k])
            idx = torch.remainder(mixed, self.buckets[n]).long()
            vectors.append(self.tables[f"n{n}_h{h}"](idx))
        return torch.cat(vectors, dim=-1)

    def forward(self, hidden_states, input_ids, valid_mask=None):
        if valid_mask is not None and valid_mask.shape != input_ids.shape:
            raise RuntimeError("Engram attention mask and input_ids shapes differ")
        memory = torch.cat([self._memory_for_order(input_ids, n) for n in self.orders], dim=-1)
        if valid_mask is not None:
            memory = memory * valid_mask.to(memory.dtype).unsqueeze(-1)
        query = self.query_norm(hidden_states.detach().float())
        key = self.key_norm(self.key_proj(memory.float()))
        gate = (key * query).sum(dim=-1) / math.sqrt(self.hidden_size)
        gate = gate.abs().clamp_min(1e-6).sqrt() * gate.sign()
        value = self.value_proj(memory.float()) * torch.sigmoid(gate).unsqueeze(-1)
        conv = self.short_conv(value.transpose(1, 2))[..., :value.shape[1]].transpose(1, 2)
        return value + conv


In [23]:
PAD_ID = tokenizer.pad_token_id
if PAD_ID is None:
    eos_id = tokenizer.eos_token_id
    PAD_ID = int(eos_id[0] if isinstance(eos_id, (tuple, list)) else eos_id)

def make_runtime():
    return {"input_ids": None, "attention_mask": None, "labels": None, "enabled": True,
            "inference_start": None, "wrapper_calls": 0, "last_delta_requires_grad": None,
            "last_delta": None, "last_hidden_shape": None, "last_ids_shape": None}

def _layer_hidden(output):
    if torch.is_tensor(output): return output, None
    if isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        return output[0], output
    raise RuntimeError(f"Unsupported decoder layer output type: {type(output)!r}")

class EngramInjectedDecoderLayer(nn.Module):
    """Real layer replacement; preserves tuple output used by Transformers."""
    def __init__(self, base_layer, engram, layer_id, runtime):
        super().__init__()
        self.base_layer, self.engram = base_layer, engram
        self.layer_id, self.runtime = int(layer_id), runtime

    def forward(self, *args, **kwargs):
        raw = self.base_layer(*args, **kwargs)
        hidden, sequence = _layer_hidden(raw)
        self.runtime["wrapper_calls"] += 1
        if not self.runtime["enabled"]: return raw
        ids, mask = self.runtime["input_ids"], self.runtime["attention_mask"]
        if not torch.is_tensor(ids):
            raise RuntimeError(f"Layer {self.layer_id}: pre-hook did not capture input_ids")
        if hidden.ndim != 3 or ids.ndim != 2 or hidden.shape[:2] != ids.shape:
            raise RuntimeError(f"Layer {self.layer_id}: hidden/IDs misaligned: hidden={tuple(hidden.shape)} ids={tuple(ids.shape)}")
        if mask is not None and mask.shape != ids.shape:
            raise RuntimeError(f"Layer {self.layer_id}: attention mask shape {tuple(mask.shape)} != ids {tuple(ids.shape)}")
        delta = self.engram(hidden, ids, valid_mask=mask)
        labels = self.runtime["labels"]
        if labels is not None:
            if labels.shape != ids.shape:
                raise RuntimeError(f"Layer {self.layer_id}: labels shape != ids shape")
            target_mask = torch.zeros_like(labels, dtype=torch.bool)
            if labels.shape[1] > 1: target_mask[:, :-1] = labels[:, 1:].ne(-100)
            delta = delta * target_mask.unsqueeze(-1)
        elif self.runtime["inference_start"] is not None:
            pos = torch.arange(hidden.shape[1], device=hidden.device)
            delta = delta * pos.ge(int(self.runtime["inference_start"])).view(1, -1, 1)
        self.runtime["last_delta"], self.runtime["last_delta_requires_grad"] = delta, bool(delta.requires_grad)
        self.runtime["last_hidden_shape"], self.runtime["last_ids_shape"] = tuple(hidden.shape), tuple(ids.shape)
        updated = hidden + delta.to(hidden.dtype)
        if sequence is None: return updated
        sequence = list(sequence); sequence[0] = updated
        return tuple(sequence) if isinstance(raw, tuple) else sequence

def _register_thinker_prehook(target_model, runtime):
    old = getattr(target_model, "_thai_engram_prehook_handle", None)
    if old is not None: old.remove()
    def _hook(module, args, kwargs):
        ids = kwargs.get("input_ids")
        if ids is None and args and torch.is_tensor(args[0]): ids = args[0]
        runtime["input_ids"], runtime["attention_mask"], runtime["labels"] = ids, kwargs.get("attention_mask"), kwargs.get("labels")
        runtime["wrapper_calls"], runtime["last_delta"] = 0, None
    target_model._thai_engram_prehook_handle = target_model.thinker.register_forward_pre_hook(_hook, with_kwargs=True)

def assert_parameter_policy(target_model):
    bad = [(n, p) for n, p in target_model.named_parameters() if p.requires_grad and ".engram." not in n]
    good = [p for n, p in target_model.named_parameters() if p.requires_grad and ".engram." in n]
    if bad: raise AssertionError(f"Frozen Qwen policy violated: {bad[:2]}")
    if not good: raise AssertionError("No Engram parameters are trainable")
    return sum(p.numel() for p in good)

def install_engram(target_model, target_processor, config, runtime):
    layers = target_model.thinker.model.layers
    hidden_size = infer_hidden_size_for(target_model, layers)
    pad = target_processor.tokenizer.pad_token_id
    if pad is None:
        eos = target_processor.tokenizer.eos_token_id
        pad = int(eos[0] if isinstance(eos, (tuple, list)) else eos)
    _register_thinker_prehook(target_model, runtime)
    wrappers = {}
    for layer_id in config["layers"]:
        layer_id = int(layer_id)
        if not 0 <= layer_id < len(layers): raise ValueError(f"Invalid layer {layer_id}")
        base = layers[layer_id]
        if isinstance(base, EngramInjectedDecoderLayer): base = base.base_layer
        engram = ThaiNgramEngram(hidden_size, config["buckets"], config["memory_dim"],
                                 config["heads"], config["kernel"], pad,
                                 config["seed"] + layer_id * 10007).to(device)
        wrappers[layer_id] = EngramInjectedDecoderLayer(base, engram, layer_id, runtime).to(device)
        layers[layer_id] = wrappers[layer_id]
    for p in target_model.parameters(): p.requires_grad_(False)
    for wrapper in wrappers.values():
        for p in wrapper.engram.parameters(): p.requires_grad_(True)
    target_model._thai_engram_wrappers = wrappers
    return wrappers, hidden_size, assert_parameter_policy(target_model)

RUNTIME = make_runtime()
ENGRAM_CONFIG = {"layers": list(ENGRAM_LAYERS), "buckets": dict(ENGRAM_BUCKETS),
                 "memory_dim": ENGRAM_DIM, "heads": ENGRAM_HEADS,
                 "kernel": ENGRAM_KERNEL, "seed": SEED}
ENGRAM_WRAPPERS, HIDDEN_SIZE, ENGRAM_TRAINABLE = install_engram(model, processor, ENGRAM_CONFIG, RUNTIME)
print(f"Engram trainable params: {ENGRAM_TRAINABLE:,}")
for wrapper in ENGRAM_WRAPPERS.values():
    assert torch.count_nonzero(wrapper.engram.value_proj.weight).item() == 0
    assert torch.count_nonzero(wrapper.engram.short_conv.weight).item() == 0


Engram trainable params: 8,249,344


## 4. Load Step-1000 Engram checkpoint


In [24]:
def _load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def validate_checkpoint(payload):
    if payload.get("base_model") != MODEL_ID:
        raise ValueError(
            f"Checkpoint base model mismatch: {payload.get('base_model')} != {MODEL_ID}"
        )

    cfg = payload.get("config", {})
    expected = {
        "layers": ENGRAM_CONFIG["layers"],
        "buckets": ENGRAM_CONFIG["buckets"],
        "memory_dim": ENGRAM_CONFIG["memory_dim"],
        "heads": ENGRAM_CONFIG["heads"],
        "kernel": ENGRAM_CONFIG["kernel"],
        "hidden_size": HIDDEN_SIZE,
    }
    for key, value in expected.items():
        if cfg.get(key) != value:
            raise ValueError(
                f"Checkpoint config mismatch for {key}: "
                f"{cfg.get(key)!r} != {value!r}"
            )

    expected_states = {str(i) for i in ENGRAM_WRAPPERS}
    if set(payload.get("engram_state", {})) != expected_states:
        raise ValueError("Checkpoint Engram layer state does not match this architecture")

def load_engram_checkpoint(step):
    """Reload one checkpoint step's weights into the already-installed Engram wrappers.

    The wrapper structure (install_engram) is built exactly once earlier in the
    notebook; this only swaps the Engram sub-module weights, so it is safe and
    cheap to call once per checkpoint step in a loop.
    """
    path = CHECKPOINT_PATHS[step]
    payload = _load_checkpoint(path)
    validate_checkpoint(payload)

    if int(payload.get("step", -1)) != step:
        raise ValueError(
            f"Expected step {step}, checkpoint reports {payload.get('step')}"
        )

    for i, wrapper in ENGRAM_WRAPPERS.items():
        wrapper.engram.load_state_dict(
            payload["engram_state"][str(i)],
            strict=True,
        )
        wrapper.engram.to(device)
        wrapper.engram.eval()

    assert_parameter_policy(model)
    return payload

print("load_engram_checkpoint(step) ready for steps:", CHECKPOINT_STEPS)
print("Engram trainable parameter count:", f"{ENGRAM_TRAINABLE:,}")

Loaded checkpoint format: qwen3-asr-thai-engram-v3
Loaded checkpoint step: 1000
Engram trainable parameter count: 8,249,344
PASS: Step-1000 Engram loaded; Qwen backbone remains frozen.


## 5. Recreate the exact fixed held-out 300-sample set

This uses the same far-tail evaluation block as the Colab scaling experiment:

```text
Porjai train stream
→ skip first 250,000 source rows
→ filter valid 1–8 second samples
→ take first 300 valid records
```

This preserves comparability with the previous step-300/500/750 results.


In [25]:
THAI_CHAR = r"\u0E00-\u0E7F"

def normalize_thai_transcript(text: str) -> str:
    text = str(text or "").strip()
    text = re.sub(r"\s+", " ", text)
    # Remove segmentation spaces only when both neighboring characters are Thai.
    text = re.sub(
        rf"(?<=[{THAI_CHAR}])\s+(?=[{THAI_CHAR}])",
        "",
        text,
    )
    return text.strip()

def get_record_text(ex):
    return normalize_thai_transcript(
        ex.get("text", ex.get("sentence", ""))
    )

def open_stream():
    ds = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
    )
    # Avoid torchcodec dependency and decode bytes ourselves.
    try:
        ds = ds.cast_column("audio", Audio(decode=False))
    except Exception as exc:
        print("Audio(decode=False) warning:", exc)
    return ds

def decode_audio(audio_obj, target_sr=16000):
    wav = None
    sr = None

    if isinstance(audio_obj, dict):
        if audio_obj.get("bytes") is not None:
            wav, sr = sf.read(io.BytesIO(audio_obj["bytes"]), dtype="float32")
        elif audio_obj.get("array") is not None:
            wav = np.asarray(audio_obj["array"], dtype=np.float32)
            sr = int(audio_obj.get("sampling_rate", target_sr))
        elif audio_obj.get("path"):
            wav, sr = sf.read(audio_obj["path"], dtype="float32")

    elif isinstance(audio_obj, (str, Path)):
        wav, sr = sf.read(str(audio_obj), dtype="float32")

    elif hasattr(audio_obj, "get_all_samples"):
        samples = audio_obj.get_all_samples()
        data = samples.data
        if torch.is_tensor(data):
            data = data.detach().cpu().numpy()
        wav = np.asarray(data, dtype=np.float32)
        sr = int(samples.sample_rate)

    if wav is None:
        raise TypeError(f"Unsupported audio object: {type(audio_obj)}")

    wav = np.asarray(wav, dtype=np.float32)

    if wav.ndim > 1:
        # Detect [channels, samples] versus [samples, channels].
        if wav.shape[0] <= 8 and wav.shape[0] < wav.shape[-1]:
            wav = wav.mean(axis=0)
        else:
            wav = wav.mean(axis=-1)

    wav = wav.reshape(-1)

    if int(sr) != target_sr:
        wav = librosa.resample(
            wav,
            orig_sr=int(sr),
            target_sr=target_sr,
        )

    return wav.astype(np.float32)

def prepare_record(ex):
    text = get_record_text(ex)
    if not text:
        return None

    wav = decode_audio(ex["audio"], target_sr=16000)
    duration = len(wav) / 16000.0

    if not (MIN_AUDIO_SEC <= duration <= MAX_AUDIO_SEC):
        return None

    return {
        "audio_array": wav,
        "text": text,
        "duration": duration,
    }

EVAL_SAMPLES = 5

# Collect a fixed held-out set from the head of the stream.
eval_records = []
eval_source_consumed = 0

for ex in open_stream():
    eval_source_consumed += 1
    rec = prepare_record(ex)
    if rec is not None:
        eval_records.append(rec)
    if len(eval_records) >= EVAL_SAMPLES:
        break

print("Held-out records:", len(eval_records))
print("Source rows consumed:", eval_source_consumed)
print("Example target:", eval_records[0]["text"])
print("Duration:", round(eval_records[0]["duration"], 2), "sec")


Held-out records: 5
Source rows consumed: 6
Example target: ทีมจากอิสราเอลไม่ควรได้เป็นเจ้าบ้านในเกมยูฟ่าคัพ
Duration: 6.4 sec


In [26]:
def collect_scaling_eval_records():
    records = []
    source_seen = 0
    ds = open_stream().skip(SCALING_EVAL_SOURCE_SKIP)

    for ex in ds:
        source_seen += 1
        try:
            rec = prepare_record(ex)
        except Exception:
            rec = None

        if rec is not None:
            records.append(rec)

        if len(records) >= SCALING_EVAL_SAMPLES:
            break

    if len(records) < SCALING_EVAL_SAMPLES:
        raise RuntimeError(
            f"Only collected {len(records)} records; wanted {SCALING_EVAL_SAMPLES}"
        )

    print(
        "Final held-out:",
        len(records),
        "records from source offset",
        SCALING_EVAL_SOURCE_SKIP,
        "(rows scanned:", source_seen, ")",
    )
    return records

scaling_eval_records = collect_scaling_eval_records()


Final held-out: 300 records from source offset 250000 (rows scanned: 421 )


## 6. Inference helpers


In [27]:
def build_prefix_text(prompt=""):
    messages = [
        {"role": "system", "content": prompt or ""},
        {"role": "user", "content": [{"type": "audio", "audio": None}]},
    ]
    rendered = processor.apply_chat_template(
        [messages],
        add_generation_prompt=True,
        tokenize=False,
    )
    return rendered[0] if isinstance(rendered, (list, tuple)) else rendered

PREFIX_TEXT = build_prefix_text("")

def move_batch(batch):
    return {
        k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v)
        for k, v in batch.items()
    }

@contextmanager
def engram_mode(enabled):
    old = RUNTIME["enabled"]
    RUNTIME["enabled"] = bool(enabled)
    try:
        yield
    finally:
        RUNTIME["enabled"] = old

def clean_generated_asr_text(decoded):
    text = decoded
    if "<asr_text>" in text:
        text = text.split("<asr_text>", 1)[1]
    eos = tokenizer.eos_token or ""
    if eos:
        text = text.replace(eos, "")
    return re.sub(r"<\|[^>]+\|>", "", text).strip()

@torch.no_grad()
def transcribe_record(record, use_engram=True):
    model.eval()
    for wrapper in ENGRAM_WRAPPERS.values():
        wrapper.engram.eval()

    inputs = processor(
        text=[PREFIX_TEXT],
        audio=[record["audio_array"]],
        return_tensors="pt",
        padding=True,
        truncation=False,
    )
    inputs = move_batch(inputs)
    inputs = {
        k: (
            v.to(dtype=MODEL_DTYPE)
            if torch.is_tensor(v) and v.is_floating_point()
            else v
        )
        for k, v in inputs.items()
    }

    RUNTIME["inference_start"] = int(inputs["input_ids"].shape[1] - 1)

    with engram_mode(use_engram):
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=GENERATION_USE_CACHE,
        )

    seq = generated.sequences if hasattr(generated, "sequences") else generated
    ids = seq[:, inputs["input_ids"].shape[1]:]

    return clean_generated_asr_text(
        tokenizer.batch_decode(
            ids,
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )[0]
    )

THAI_CHAR_RE = r"\u0E00-\u0E7F"
def thai_cer_norm(text):
    return re.sub(r"\s+", "", normalize_thai_transcript(text).lower())

def corpus_cer(refs, preds):
    ref = "".join(thai_cer_norm(x) for x in refs)
    pred = "".join(thai_cer_norm(x) for x in preds)
    return cer(ref, pred)

print("Inference ready.")
print("use_cache:", GENERATION_USE_CACHE)


Inference ready.
use_cache: False


## 7. FINAL CELL — Evaluate Base vs Engram across steps 300 → 1000

This is the only expensive stage in this notebook.

It generates:
- Base predictions for the same 300 samples (computed once, cached)
- Engram predictions for the same 300 samples, at every checkpoint step in `CHECKPOINT_STEPS` (300, 500, 750, 1000)

Each step reloads that checkpoint's weights into the already-installed Engram wrappers before generating, so the model/architecture is built exactly once and only the Engram weights change between steps.

Per-step results are saved individually, and a combined `scaling_300_to_1000_summary.json` is written at the end covering the whole 300→1000 CER trend.

The Base predictions are cached locally, so rerunning this cell after an interruption does not regenerate the Base half.

In [28]:
BASE_CACHE = OUTPUT_DIR / "fixed300_base_predictions.json"

refs = [r["text"] for r in scaling_eval_records]

# Cache Base predictions because these never depend on the Engram checkpoint.
if BASE_CACHE.exists():
    cached = json.loads(BASE_CACHE.read_text(encoding="utf-8"))
    if (
        cached.get("source_skip") == SCALING_EVAL_SOURCE_SKIP
        and len(cached.get("predictions", [])) == len(refs)
        and cached.get("references") == refs
    ):
        base_preds = cached["predictions"]
        print("Reused cached Base predictions:", BASE_CACHE)
    else:
        print("Base cache does not match this fixed set; regenerating.")
        base_preds = None
else:
    base_preds = None

if base_preds is None:
    print("Generating Base predictions...")
    base_preds = [
        transcribe_record(record, False)
        for record in tqdm(scaling_eval_records, desc="Base 300-sample eval")
    ]
    BASE_CACHE.write_text(
        json.dumps(
            {
                "source_skip": SCALING_EVAL_SOURCE_SKIP,
                "references": refs,
                "predictions": base_preds,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Saved Base cache:", BASE_CACHE)

base_per_sample = [
    cer(thai_cer_norm(ref), thai_cer_norm(pred))
    for ref, pred in zip(refs, base_preds)
]
base_mean = float(np.mean(base_per_sample))
base_corpus = float(corpus_cer(refs, base_preds))

all_steps_summary = []
per_step_rows = {}

for step in CHECKPOINT_STEPS:
    print(f"\n=== Step {step}: loading checkpoint and generating Engram predictions ===")
    payload = load_engram_checkpoint(step)

    step_json = OUTPUT_DIR / f"preset_a_layer2_step{step}_evaluation.json"
    step_csv = OUTPUT_DIR / f"preset_a_layer2_step{step}_predictions.csv"

    engram_preds = [
        transcribe_record(record, True)
        for record in tqdm(scaling_eval_records, desc=f"Engram step {step}")
    ]
    engram_per_sample = [
        cer(thai_cer_norm(ref), thai_cer_norm(pred))
        for ref, pred in zip(refs, engram_preds)
    ]
    engram_mean = float(np.mean(engram_per_sample))
    engram_corpus = float(corpus_cer(refs, engram_preds))

    improved = sum(e < b for e, b in zip(engram_per_sample, base_per_sample))
    tied = sum(e == b for e, b in zip(engram_per_sample, base_per_sample))
    regressed = len(refs) - improved - tied

    summary = {
        "step": step,
        "samples": len(refs),
        "base_mean_sample_cer": base_mean,
        "engram_mean_sample_cer": engram_mean,
        "relative_mean_sample_cer_improvement_percent": (
            (base_mean - engram_mean) / base_mean * 100.0
            if base_mean else None
        ),
        "base_corpus_cer": base_corpus,
        "engram_corpus_cer": engram_corpus,
        "relative_corpus_cer_improvement_percent": (
            (base_corpus - engram_corpus) / base_corpus * 100.0
            if base_corpus else None
        ),
        "improved_samples": improved,
        "tied_samples": tied,
        "regressed_samples": regressed,
        "checkpoint": str(CHECKPOINT_PATHS[step]),
        "checkpoint_format": payload.get("format"),
        "gpu": torch.cuda.get_device_name(0),
        "rocm_hip": torch.version.hip,
        "generation_use_cache": GENERATION_USE_CACHE,
        "eval_source_skip": SCALING_EVAL_SOURCE_SKIP,
    }

    rows = []
    for i, (ref, base, eng, bc, ec) in enumerate(
        zip(refs, base_preds, engram_preds, base_per_sample, engram_per_sample)
    ):
        rows.append(
            {
                "index": i,
                "reference": ref,
                "base": base,
                f"engram_step{step}": eng,
                "base_cer": float(bc),
                "engram_cer": float(ec),
                "cer_delta": float(ec - bc),
                "improved": bool(ec < bc),
            }
        )

    pd.DataFrame(rows).to_csv(step_csv, index=False)
    step_json.write_text(
        json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    all_steps_summary.append(summary)
    per_step_rows[step] = rows

    print(
        f"Step {step} DONE — corpus CER {base_corpus:.4f} -> {engram_corpus:.4f} "
        f"({summary['relative_corpus_cer_improvement_percent']:.1f}% rel. improvement)"
    )
    print("Saved:", step_csv)
    print("Saved:", step_json)

SCALING_SUMMARY_JSON = OUTPUT_DIR / "scaling_300_to_1000_summary.json"
SCALING_SUMMARY_JSON.write_text(
    json.dumps({"steps": CHECKPOINT_STEPS, "results": all_steps_summary}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nFINAL SCALING SUMMARY (300 -> 1000)")
display(pd.DataFrame(all_steps_summary)[
    [
        "step",
        "base_corpus_cer",
        "engram_corpus_cer",
        "relative_corpus_cer_improvement_percent",
        "improved_samples",
        "tied_samples",
        "regressed_samples",
    ]
])
print("Saved:", SCALING_SUMMARY_JSON)

Generating Base predictions...


Base 300-sample eval:   0%|          | 0/300 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
/home/cepheusn/Developments/Engram-qwenASR/.venv/lib64/python3.14/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: 1Torch was not compiled with memory efficient attention. (Triggered internally at /builddir/build/BUILD/python-torch-2.9.1-build/pytorch-v2.9.1/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:778.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Base 300-sample eval: 100%|██████████| 300/300 [04:28<00:00,  1.12it/s]


Saved Base cache: /home/cepheusn/Developments/Engram-qwenASR/Qwen3ASR_Thai_Engram_ROCm/outputs/fixed300_base_predictions.json
Generating Step-1000 Engram predictions...


Engram step 1000: 100%|██████████| 300/300 [04:50<00:00,  1.03it/s]


FINAL STEP-1000 RESULT
{
  "step": 1000,
  "samples": 300,
  "base_mean_sample_cer": 0.19197821025998418,
  "engram_mean_sample_cer": 0.1458730533193524,
  "relative_mean_sample_cer_improvement_percent": 24.015828087049265,
  "base_corpus_cer": 0.17805051583066525,
  "engram_corpus_cer": 0.11339381003201708,
  "relative_corpus_cer_improvement_percent": 36.31368631368631,
  "improved_samples": 133,
  "tied_samples": 110,
  "regressed_samples": 57,
  "checkpoint": "/home/cepheusn/Developments/Engram-qwenASR/checkpoints/engram_step_001000.pt",
  "checkpoint_format": "qwen3-asr-thai-engram-v3",
  "gpu": "AMD Radeon RX 9070 XT",
  "rocm_hip": "7.1.52802-9999",
  "generation_use_cache": false,
  "eval_source_skip": 250000
}

Saved: /home/cepheusn/Developments/Engram-qwenASR/Qwen3ASR_Thai_Engram_ROCm/outputs/preset_a_layer2_step1000_final_predictions.csv
Saved: /home/cepheusn/Developments/Engram-qwenASR/Qwen3ASR_Thai_Engram_ROCm/outputs/preset_a_layer2_step1000_final_evaluation.json


,step,samples,base_mean_sample_cer,engram_mean_sample_cer,relative_mean_sample_cer_improvement_percent,base_corpus_cer,engram_corpus_cer,relative_corpus_cer_improvement_percent,improved_samples,tied_samples,regressed_samples,checkpoint,checkpoint_format,gpu,rocm_hip,generation_use_cache,eval_source_skip
0,1000,300,0.191978,0.145873,24.015828,0.178051,0.113394,36.313686,133,110,57,/home/cepheusn/Developments/Engram-qwenASR/che...,qwen3-asr-thai-engram-v3,AMD Radeon RX 9070 XT,7.1.52802-9999,False,250000


## Expected outcome

Take the Step-1000 result from the final cell and compare it directly with the already completed scaling points:

```text
Base
Step 300
Step 500
Step 750
Step 1000  ← this notebook
```

The key question is whether Step 1000:
- continues improving,
- plateaus,
- or regresses relative to the earlier checkpoints.

This notebook intentionally contains **zero training code**.
